# 04 — Innovation Retrieval

## Purpose

Use semantic similarity to identify project components and subcomponents that
may contain innovation signals.

This notebook:

1. Loads precomputed chunk embeddings and metadata.
2. Encodes a small set of innovation-oriented search queries.
3. Retrieves semantically similar project text.
4. Consolidates duplicate results at the analytical-unit level.
5. Saves a compact candidate set for structured innovation classification.

Semantic retrieval is used only for candidate discovery. It does not determine
whether an activity is innovative.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer


# --------------------------------------------------
# PROJECT CONFIGURATION
# --------------------------------------------------

ROOT = Path("..").resolve()

DATA_DIR = ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"

EMBEDDINGS_PATH = PROCESSED_DIR / "pad_chunk_embeddings.npy"
CHUNK_METADATA_PATH = PROCESSED_DIR / "pad_chunk_metadata.csv"
CANDIDATES_PATH = PROCESSED_DIR / "innovation_candidates.csv"

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
TOP_K_PER_QUERY = 25


# --------------------------------------------------
# LOAD EMBEDDINGS AND METADATA
# --------------------------------------------------

embeddings = np.load(EMBEDDINGS_PATH)
chunks_df = pd.read_csv(CHUNK_METADATA_PATH)

assert len(chunks_df) == len(embeddings), (
    "Embedding rows and metadata rows do not align."
)

print("Chunks     :", len(chunks_df))
print("Embeddings :", embeddings.shape)


In [ ]:
# --------------------------------------------------
# LOAD EMBEDDING MODEL AND DEFINE RETRIEVAL FUNCTION
# --------------------------------------------------

model = SentenceTransformer(EMBEDDING_MODEL)

print("Embedding model:", EMBEDDING_MODEL)


def semantic_search(query, top_k=TOP_K_PER_QUERY):
    """
    Retrieve the most semantically similar embedded chunks for a query.
    """
    query_embedding = model.encode(
        query,
        normalize_embeddings=True
    )

    scores = embeddings @ query_embedding

    top_indices = np.argsort(scores)[::-1][:top_k]

    results = chunks_df.iloc[top_indices].copy()
    results["similarity"] = scores[top_indices]
    results["query"] = query

    return results

In [ ]:
# --------------------------------------------------
# INNOVATION QUERIES AND RETRIEVAL
# --------------------------------------------------

INNOVATION_QUERIES = {
    "digital_data": """
        use of digital technologies, satellite data, remote sensing,
        geospatial information, data platforms, digital systems, or
        information systems to improve monitoring, planning, or decision-making
    """,
    "institutional_process": """
        new institutional arrangement, coordination mechanism, governance model,
        service delivery mechanism, implementation model, or organizational reform
    """,
    "financing_delivery": """
        innovative financing mechanism, results-based financing, matching grants,
        performance-based payments, risk sharing, blended finance, or new delivery model
    """,
    "pilot_novelty": """
        pilot, demonstration, proof of concept, first-of-its-kind approach,
        new technology, new method, or experimental intervention
    """,
    "scaling_replication": """
        scaling up, replication, expansion, mainstreaming, adoption,
        or transfer of an approach previously tested elsewhere
    """,
}

# --------------------------------------------------
# RETRIEVE CANDIDATE CHUNKS
# --------------------------------------------------

retrieved_frames = []

for query_name, query_text in INNOVATION_QUERIES.items():

    query_results = semantic_search(
        query_text,
        top_k=TOP_K_PER_QUERY
    )

    query_results["retrieval_query"] = query_name

    retrieved_frames.append(query_results)

retrieved_df = pd.concat(
    retrieved_frames,
    ignore_index=True
)

print("Raw retrieved chunks:", len(retrieved_df))

In [ ]:
# --------------------------------------------------
# DEDUPLICATE AND SAVE
# --------------------------------------------------

innovation_candidates_df = (
    retrieved_df
    .sort_values("similarity", ascending=False)
    .drop_duplicates(subset=["projectid", "type", "number"], keep="first")
    .reset_index(drop=True)
)

print("Unique candidate units:", len(candidate_units_df))
print()
display(
    candidate_units_df[
        ["projectid", "type", "number", "title", "retrieval_query", "similarity"]
    ].head(20)
)

innovation_candidates_df.to_csv(
    PROCESSED_DIR / "innovation_candidates.csv", index=False
)
print("Saved: innovation_candidates.csv")

In [ ]:
innovation_candidates_df[
    [
        "projectid",
        "type",
        "number",
        "title",
        "retrieval_query",
        "similarity",
    ]
].head(10)